<div style='float: right'><img src='pic/mini_sudoku.png'/></div>

## <div id='mini_sudoku' />ミニ数独

In [ ]:
import numpy as np
from mip import Model, xsum

data = [
    [0, 5, 0, 3, 0, 0],
    [0, 0, 3, 1, 0, 0],
    [0, 0, 4, 0, 0, 0],
    [0, 6, 0, 0, 4, 0],
    [0, 0, 1, 0, 0, 3],
    [0, 0, 0, 6, 0, 0]
]

### 問題
* 空いているマスに1から6の数字を入れてください
* 1行に同じ数字は1つ
* 1列に同じ数字は1つ
* 2行3列のブロックに同じ数字は1つ

### 変数
* v：行ごと列ごと数ごとの0-1変数

### 制約
* 固定の数字
* 1マスに数字は1つ
* 1行に同じ数字は1つ
* 1列に同じ数字は1つ
* 1ブロックに同じ数字は1つ

In [ ]:
m = Model()
v = m.add_var_tensor((6, 6, 6), "v", var_type="B")  # y, x, n
for y in range(6):
    for x in range(6):
        if data[y][x]:
            v[y, x, data[y][x]-1].lb = 1  # 固定
        m += xsum(v[y, x, :]) == 1  # 1マスに数字は1つ
for n in range(6):
    for i in range(6):
        m += xsum(v[i, :, n]) == 1  # 1行に同じ数字は1つ
        m += xsum(v[:, i, n]) == 1  # 1列に同じ数字は1つ
    for y in range(0, 6, 2):
        for x in range(0, 6, 3):
            # 1ブロックに同じ数字は1つ
            m += xsum(v[y:y+2, x:x+3, n].flat) == 1
m.verbose = 0
m.optimize()
ans = v.astype(float, subok=False).round().astype(int)
print((ans * range(1,7)).sum(axis=2))